# From Pilot to Payoff - 07: Conclusion & Summary

## Setup and data preparation

*Re-runs the shared imports, data loading, feature engineering, and standardisation so the notebook is self-contained. Identical across the analysis modules; see `01_setup_and_data_prep.ipynb` for the documented version with the data dictionary.*

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

import statsmodels.api as sm
from statsmodels.formula.api import ols, logit
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

import matplotlib.pyplot as plt
import seaborn as sns

# Folder for saving generated figures.
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)


In [2]:
company = pd.read_csv('ai_company_adoption.csv')
country_index = pd.read_csv('country_ai_index.csv')
industry_summary = pd.read_csv('ai_industry_summary.csv')

print('Dataset shapes')
print(f'company adoption: {company.shape[0]:,} rows x {company.shape[1]:,} columns')
print(f'country index:    {country_index.shape[0]:,} rows x {country_index.shape[1]:,} columns')
print(f'industry summary: {industry_summary.shape[0]:,} rows x {industry_summary.shape[1]:,} columns')

company.head(3)


Dataset shapes
company adoption: 150,000 rows x 43 columns
country index:    30 rows x 8 columns
industry summary: 9 rows x 8 columns


,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method
0,1,COMP-00001,2023,Q1,Italy,Europe,Education,Startup,57,48.31,...,2.65,1,1,3,2.52,9.45,53,5.20,WEF Survey,API Scrape
1,2,COMP-00001,2023,Q2,Italy,Europe,Education,Startup,57,48.31,...,5.77,2,2,5,4.77,0.00,51,6.98,McKinsey Report,Phone Interview
2,3,COMP-00001,2023,Q3,Italy,Europe,Education,Startup,57,48.31,...,6.94,3,3,2,12.87,9.74,40,4.12,Internal Corporate Survey,Research Compilation


In [3]:
# Feature engineering and merge.
quarter_map = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
company = company.copy()
company['quarter_num'] = company['quarter'].map(quarter_map)
company['advanced_adoption'] = company['ai_adoption_stage'].isin(['partial', 'full']).astype(int)
company['net_jobs_created'] = company['jobs_created'] - company['jobs_displaced']
company['net_jobs_per_100_employees'] = company['net_jobs_created'] / company['num_employees'] * 100
company['reskilling_rate_per_100_employees'] = company['reskilled_employees'] / company['num_employees'] * 100

# Log transform for the skewed investment variable. log1p handles zero values safely.
company['log_ai_investment_per_employee'] = np.log1p(company['ai_investment_per_employee'])

country_model = country_index.drop(columns=['region'])
country_model['log_gdp_per_capita'] = np.log1p(country_model['gdp_per_capita'])
country_model['log_ai_patent_filings_2024'] = np.log1p(country_model['ai_patent_filings_2024'])

df = company.merge(country_model, on='country', how='left')

# Latest observation per company for robustness checks.
latest_obs = (df.sort_values(['company_id', 'survey_year', 'quarter_num'])
                .groupby('company_id', as_index=False)
                .tail(1)
                .reset_index(drop=True))

# Digital maturity tertiles for interaction models.
df['digital_maturity_tertile'] = pd.qcut(
    df['digital_maturity_index'], q=3, labels=['Low', 'Medium', 'High']
)
latest_obs['digital_maturity_tertile'] = pd.qcut(
    latest_obs['digital_maturity_index'], q=3, labels=['Low', 'Medium', 'High']
)

merge_check = df[['country', 'digital_maturity_index']].isna().mean().rename('missing_rate')
print(f'Merged modelling data: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Latest-company robustness data: {latest_obs.shape[0]:,} rows')
print('\nCountry-index missing check:')
print(merge_check)


Merged modelling data: 150,000 rows x 58 columns
Latest-company robustness data: 10,000 rows

Country-index missing check:
country                   0.0
digital_maturity_index    0.0
Name: missing_rate, dtype: float64


In [4]:
# Standardise numeric predictors used in regression models.
standardise_cols = [
    'ai_training_hours', 'num_ai_tools_used', 'ai_projects_active', 'years_using_ai',
    'ai_budget_percentage', 'log_ai_investment_per_employee',
    'regulatory_compliance_score', 'ai_risk_management_score'
]

for col in standardise_cols:
    mean = df[col].mean()
    std = df[col].std(ddof=0)
    df[f'z_{col}'] = (df[col] - mean) / std
    latest_obs[f'z_{col}'] = (latest_obs[col] - mean) / std

print('Created z-score variables for regression comparability.')


Created z-score variables for regression comparability.


In [5]:
stage_order = ['none', 'pilot', 'partial', 'full']

## Recompute headline metrics

*Self-contained re-derivation of the five numbers used in the evidence table (Q1-Q5). The nested logits and business OLS are re-run here.*

In [6]:
# Recompute the headline metrics used in the evidence table below.
# This conclusion module is self-contained, so it re-derives the key numbers from Q1-Q5.
# The heavier models (nested logits, business OLS) are re-run here by design.

# Q1 - overall (pooled) advanced adoption rate
p_hat = df['advanced_adoption'].mean()

# Q2 - country readiness PC1 -> OLS adjusted R2
country_level = (df.groupby('country')
                   .agg(adv_adoption_rate=('advanced_adoption', 'mean'),
                        log_gdp_per_capita=('log_gdp_per_capita', 'first'),
                        internet_penetration=('internet_penetration', 'first'),
                        digital_maturity_index=('digital_maturity_index', 'first'),
                        log_ai_patent_filings_2024=('log_ai_patent_filings_2024', 'first'),
                        ai_researchers_per_million=('ai_researchers_per_million', 'first'))
                   .reset_index())
readiness_features = ['log_gdp_per_capita', 'internet_penetration', 'digital_maturity_index',
                      'log_ai_patent_filings_2024', 'ai_researchers_per_million']
X_readiness = StandardScaler().fit_transform(country_level[readiness_features])
pca = PCA(n_components=2)
country_level['readiness_pc1'] = pca.fit_transform(X_readiness)[:, 0]
if country_level[['readiness_pc1', 'digital_maturity_index']].corr().iloc[0, 1] < 0:
    country_level['readiness_pc1'] *= -1
model_country_pc1 = ols('adv_adoption_rate ~ readiness_pc1', data=country_level).fit()

# Q3 - nested logistic models (pseudo-R2 and AUC)
control_terms = 'C(country) + C(industry) + C(company_size) + C(survey_year) + C(quarter)'
capability_terms = 'z_ai_training_hours + z_num_ai_tools_used + z_ai_projects_active + z_years_using_ai'
investment_terms = 'z_ai_budget_percentage + z_log_ai_investment_per_employee'
governance_terms = 'z_regulatory_compliance_score + z_ai_risk_management_score + C(data_privacy_level) + C(ai_ethics_committee)'
logit_specs = {
    'M0 Controls only': control_terms,
    'M1 Controls + Capability': control_terms + ' + ' + capability_terms,
    'M2 Controls + Investment': control_terms + ' + ' + investment_terms,
    'M3 Controls + Governance': control_terms + ' + ' + governance_terms,
    'M4 Full driver model': control_terms + ' + ' + capability_terms + ' + ' + investment_terms + ' + ' + governance_terms,
}
logit_rows = []
for name, rhs in logit_specs.items():
    res = logit('advanced_adoption ~ ' + rhs, data=df).fit(method='lbfgs', maxiter=200, disp=False)
    logit_rows.append({'model': name,
                       'McFadden_pseudo_R2': res.prsquared,
                       'AUC': roc_auc_score(df['advanced_adoption'], res.predict(df))})
logit_compare = pd.DataFrame(logit_rows)

# Q4 - business OLS adjusted-R2 gain from advanced adoption (only M0/M1 needed for this metric)
ols_control_terms = control_terms
business_outcomes = {
    'productivity_change_percent': 'Productivity change (%)',
    'revenue_growth_percent': 'Revenue growth (%)',
    'cost_reduction_percent': 'Cost reduction (%)',
    'innovation_score': 'Innovation score',
    'customer_satisfaction': 'Customer satisfaction'
}
business_ols_rows = []
for outcome, label in business_outcomes.items():
    m0 = ols(f'{outcome} ~ {ols_control_terms}', data=df).fit(cov_type='cluster', cov_kwds={'groups': df['company_id']})
    m1 = ols(f'{outcome} ~ advanced_adoption + {ols_control_terms}', data=df).fit(cov_type='cluster', cov_kwds={'groups': df['company_id']})
    business_ols_rows.append({'outcome': label, 'delta_adj_R2_advanced': m1.rsquared_adj - m0.rsquared_adj})
business_ols_summary = pd.DataFrame(business_ols_rows)

# Q5 - Enterprise posterior mean (Beta-Binomial on latest observation per company)
size_counts = (latest_obs.groupby('company_size')['advanced_adoption']
                 .agg(x='sum', n='count')
                 .reindex(['Startup', 'SME', 'Enterprise']).reset_index())
posterior_rows = []
for _, row in size_counts.iterrows():
    a_post, b_post = 1 + int(row['x']), 1 + int(row['n']) - int(row['x'])
    posterior_rows.append({'company_size': row['company_size'], 'posterior_mean': a_post / (a_post + b_post)})
posterior_summary = pd.DataFrame(posterior_rows)

print('Headline metrics recomputed for the evidence table.')

Headline metrics recomputed for the evidence table.


## Section 5: Conclusion and Summary

### 5.1 Key Insights

The notebook supports five main findings:

1. **AI adoption is uneven across firm size, region, industry, and time, and is rising.** The EDA shows a large pilot/partial middle, with full adoption still rare (~1%). Advanced adoption rose steadily from ~45% (2023) to ~62% (2026), so "advanced" here means mostly *partial* adoption rather than full maturity.
2. **Country readiness is associated with advanced adoption, but the readiness indicators are highly correlated.** PCA (PC1 ~ 87% of variance) is a defensible way to summarise country readiness before regression. With only 30 countries this is exploratory.
3. **Capability and investment are the strongest firm-level driver blocks.** Nested logistic regression shows capability and investment improve pseudo-R2 and AIC/BIC substantially beyond controls, while governance adds little. In the full model the continuous governance scores are *not* independently associated with advanced adoption (OR ~ 1.0); only the ethics-committee indicator retains a modest association. Governance is best described as a responsible-AI control rather than a predictor of progression.
4. **Advanced adoption is associated with stronger business outcomes, most clearly productivity (delta adj R2 ~ 0.24).** This is an association, not proven causation, and national digital maturity does not meaningfully moderate it (the interaction is at most marginal).
5. **Advanced adoption is associated with more reskilling and markedly lower AI failure rates.** Net job creation is also positive but practically negligible (~0.5 jobs per 100 employees). The Bayesian analysis shows enterprises adopt at much higher rates than SMEs and startups, which are themselves indistinguishable.

### 5.2 Recommendations

- **For policymakers:** Improve digital infrastructure, AI policy clarity, and digital skills pipelines, especially in lower-readiness markets.
- **For business leaders:** Prioritise capability building and sustained investment, then use governance and risk controls to make adoption more reliable and responsible.
- **For AI vendors and consultants:** The large pilot-stage segment, and the gap between enterprises and SMEs/startups, suggests demand for SME-targeted implementation support, governance templates, integration services, and workforce enablement.
- **For workforce planners:** Reskilling should be built into AI deployment plans instead of being treated as a downstream reaction.

*(These are illustrative implications drawn from a simulated dataset - see 5.3 - rather than prescriptive guidance.)*

### 5.3 Limitations

- **Synthetic data.** This is a public, simulated dataset (no missing values or duplicates across 150k rows, full-year 2026 data, and `ai_adoption_stage` is an exact binning of `ai_adoption_rate`). Strong associations - e.g. the 0.93 model AUC and the large budget/training odds ratios - partly reflect the internal structure of the simulated data, not empirical real-world drivers. The project demonstrates correct statistical methodology rather than substantive global findings.
- **Panel structure.** The data are firm-quarter observations, so repeated company observations can inflate significance. This notebook uses cluster-robust standard errors, latest-company robustness checks, and a company-level hold-out where appropriate.
- The dataset is not an official global survey, so findings should be interpreted as associations, not official estimates.
- Country-level analysis has only 30 countries and should be treated as exploratory.
- 2026-labelled observations should be interpreted cautiously in time-trend analysis.
- The analysis is observational, so no causal claims are made.

### 5.4 Future Work

- Fit hierarchical or multilevel models with firm, country, and industry structure.
- Explore time-lagged models, such as whether investment in one quarter predicts adoption maturity in later quarters.
- Compare regression results with tree-based models for prediction while preserving regression for interpretation.
- Enrich country readiness with external indexes such as government AI readiness or AI infrastructure measures.
- Validate the key associations on a real (non-synthetic) AI-adoption dataset before drawing substantive conclusions.


In [7]:
# Final compact evidence table for report/slide writing.
# advanced-adoption rate recomputed here (the current Q1 cell no longer defines p_hat).
p_hat = df['advanced_adoption'].mean()
final_evidence = pd.DataFrame({
    'evidence_item': [
        'Overall advanced adoption rate',
        'Country readiness PC1 OLS adjusted R2',
        'Best logistic model pseudo-R2',
        'Best logistic model AUC',
        'Largest business adjusted R2 gain from advanced adoption',
        'Enterprise posterior mean advanced adoption (latest company observation)'
    ],
    'value': [
        f'{p_hat:.1%}',
        f'{model_country_pc1.rsquared_adj:.3f}',
        f'{logit_compare["McFadden_pseudo_R2"].max():.3f}',
        f'{logit_compare["AUC"].max():.3f}',
        f'{business_ols_summary["delta_adj_R2_advanced"].max():.3f}',
        f'{posterior_summary.loc[posterior_summary["company_size"] == "Enterprise", "posterior_mean"].iloc[0]:.1%}'
    ]
})

final_evidence


,evidence_item,value
0,Overall advanced adoption rate,53.7%
1,Country readiness PC1 OLS adjusted R2,0.311
2,Best logistic model pseudo-R2,0.505
3,Best logistic model AUC,0.927
4,Largest business adjusted R2 gain from advance...,0.244
5,Enterprise posterior mean advanced adoption (l...,78.6%
